# Credit Risk Model - Fixed Pipeline for Balanced Predictions\n
\n
Target: Increase class 1 predictions from 14 to ~100/200 on test set.

In [17]:
%pip install xgboost

import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, accuracy_score

Note: you may need to restart the kernel to use updated packages.


In [18]:
# 1. Load and preprocess data
df = pd.read_csv("credit_risk_dataset.csv")
# Fill missing values
df.fillna(df.median(numeric_only=True), inplace=True)

# Encode categoricals
le_education = LabelEncoder()
le_housing = LabelEncoder()
df['Education_Level'] = le_education.fit_transform(df['Education_Level'])
df['Housing_Status'] = le_housing.fit_transform(df['Housing_Status'])

# Features and target
X = df[['Age', 'Income', 'Loan_Amount', 'Credit_Score', 'Employment_Years', 'Education_Level', 'Housing_Status']]
y = df['Default']

print("Original distribution:", y.value_counts().sort_index())

Original distribution: Default
0    862
1    138
Name: count, dtype: int64


In [19]:
# 2. Train/test split (fixed seed for repro)\n
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# Scale\n
scaler = RobustScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print("Train shape:", X_train.shape, "y_train 1s:", np.sum(y_train))
print("Test shape:", X_test.shape)

Train shape: (800, 7) y_train 1s: 110
Test shape: (200, 7)


In [20]:
# 3. Aggressive SMOTE (1:1 balance)
smote = SMOTE(sampling_strategy=1.0, random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print("Post-SMOTE train balance:", np.bincount(y_train_sm))

Post-SMOTE train balance: [690 690]


In [21]:
# 4. Train XGBoost with imbalance handling
model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=13,  # 186/14 ≈13
    random_state=42
)
model.fit(X_train_sm, y_train_sm)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=None, num_parallel_tree=None, ...)

In [22]:
# 5. Predict and verify balance\n
pred = model.predict(X_test)
print("Test predictions balance:")
print("0 count:", np.sum(pred==0))
print("1 count:", np.sum(pred==1))
print("Accuracy:", accuracy_score(y_test, pred))
print(" \\nClassification Report:\\n", classification_report(y_test, pred))

Test predictions balance:
0 count: 170
1 count: 30
Accuracy: 0.74
 \nClassification Report:\n               precision    recall  f1-score   support

           0       0.85      0.84      0.85       172
           1       0.10      0.11      0.10        28

    accuracy                           0.74       200
   macro avg       0.48      0.48      0.48       200
weighted avg       0.75      0.74      0.74       200



In [23]:
# 6. Save model, scaler, encoders\n
pickle.dump(model, open("credit_model.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl", "wb"))
pickle.dump(le_education, open("education_encoder.pkl","wb"))
pickle.dump(le_housing, open("housing_encoder.pkl","wb"))
print("Model pipeline saved successfully!")

Model pipeline saved successfully!
